# PathSnap — Member 1 EDA
Train and student-test only. Read the measured inventory, quality, alignment and candidate-sweep outputs. AI-assisted notebook; all values come from executed local analysis. See `report/member1/README.md` for reproduction and uncertainty.

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks':
    os.chdir('..')
import pandas as pd
from IPython.display import display, Image
quality = pd.read_csv('results/metrics/quality_report.csv', dtype={'trip_id': str})
display(quality.groupby('partition')[['n_obs','n_nodes','n_arcs','route_breaks','missing_values','large_gaps']].agg(['sum','min','median','max']))

In [ ]:
inventory = pd.read_csv('results/metrics/inventory.csv', dtype={'trip_id': str})
display(inventory.groupby(['partition','file_type']).agg(files=('trip_id','count'), rows=('rows','sum'), bytes=('bytes','sum')))
display(quality.sort_values('max_gap_s', ascending=False).head(10))

## Approximate labels
Offline dynamic programming aligns GPS observations with ordered reference arc occurrences. Assignments beyond 100 m are unknown, never negative training examples. Compare with independently measured nearest-reference distances; inspect offset trips before interpreting pointwise accuracy.

In [ ]:
alignment = pd.read_csv('results/metrics/label_alignment.csv', dtype={'trip_id': str})
display(alignment.groupby('partition')[['n_obs','unknown_labels','route_breaks']].sum())
display(alignment.sort_values('unknown_labels', ascending=False).head(10))
display(pd.read_csv('results/metrics/reference_distance_diagnostics.csv', dtype={'trip_id': str}).sort_values('nearest_reference_over100m', ascending=False).head(10))

## Training-only candidate sweep
Search radius/K are selected on training observations. Known-label candidate miss rate sets the pointwise Top-1 ceiling; excluded unknown observations are reported separately. Scaling is fitted inside training folds, never during EDA.

In [ ]:
display(pd.read_csv('results/metrics/candidate_sweep.csv'))
for plot in ['eda_timestamp_gaps','eda_n_obs','eda_n_arcs','eda_missing_values','eda_connectivity','eda_trajectory','eda_candidate_sweep']:
    display(Image(filename=f'results/plots/{plot}.png'))

## Interactive train map
Open [the train GPS/network map](../maps/eda_train_00000001.html). It has separate road/GPS layers and a legend. Predictions and reference-route comparison maps are Member 3’s deliverable.

In [ ]:
# Optional full regeneration, from the repository root:
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'src.data_processing.run_member1'], check=True)